In [1]:
import copy
import os
import time
from functools import partial
from pathlib import Path

import numpy as np
import torch
import torch.ao.quantization
from transformers import AutoModel, AutoTokenizer
import warnings

torch.set_float32_matmul_precision('high')
warnings.filterwarnings("ignore", category=UserWarning, module="torch._inductor")

In [2]:
device = "cpu"
device

'cpu'

In [3]:
MODEL_NAME = "sentence-transformers/multi-qa-mpnet-base-cos-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
sentence_a = "I go to school every day because I want to learn new things."
sentence_b = "My brother likes playing computer games in his free time."
sentences = [sentence_a, sentence_b]

encodings = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(
    device
)

tokens_a = tokenizer.convert_ids_to_tokens(encodings["input_ids"][0])
print(tokens_a)

tokens_b = tokenizer.convert_ids_to_tokens(encodings["input_ids"][1])
print(tokens_b)

['<s>', 'i', 'go', 'to', 'school', 'every', 'day', 'because', 'i', 'want', 'to', 'learn', 'new', 'things', '.', '</s>']
['<s>', 'my', 'brother', 'likes', 'playing', 'computer', 'games', 'in', 'his', 'free', 'time', '.', '</s>', '<pad>', '<pad>', '<pad>']


## 1. Evaluation mode

Here, I define simple utility functions for running benchmarks and printing their results, which will be reused throughout this notebook.

In [5]:
def benchmark(
    fn,
    device,
    fn_setup=lambda: None,
    fn_cleanup=lambda: None,
    n_runs=100,
    n_warmup=10,
):
    fn_setup()

    for _ in range(n_warmup):
        fn()

    if device == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)

    fn_cleanup()
    return np.mean(times), np.std(times)


def benchmark_model(fn, model, inputs, **kwargs):
    """Benchmark `fn(model, inputs)` on the device the model currently lives on (cpu / cuda)."""
    model_device = next(model.parameters()).device.type
    assert all(v.device.type == model_device for v in inputs.values()), (
        "model and inputs are on different devices"
    )

    return benchmark(partial(fn, model, inputs), device=model_device, **kwargs)


def print_benchmark_results(results, baseline):
    baseline_mean = results[baseline][0]

    width = max(len(name) for name in results) + 2

    print(f"{'mode':<{width}}{'mean [ms]':>10}{'std [ms]':>10}{'speedup':>9}")
    for name, (mean, std) in results.items():
        print(f"{name:<{width}}{mean:>10.2f}{std:>10.2f}{baseline_mean / mean:>8.2f}x")

In [6]:
mode = "training" if model.training else "eval"
print(f"Model is currently in {mode} mode.")

Model is currently in eval mode.


In [7]:
# from_pretrained() returns the model already in eval mode.


def no_optimization(model, inputs):
    model(**inputs)


def eval_mode(model, inputs):
    model(**inputs)


def eval_mode_and_no_grad(model, inputs):
    with torch.no_grad():
        model(**inputs)


def eval_mode_and_inference_mode(model, inputs):
    with torch.inference_mode():
        model(**inputs)

In [8]:
results = {}

results["no optimization"] = benchmark_model(
    no_optimization, model, encodings, fn_setup=model.train, fn_cleanup=model.eval
)
results["no optimization"]

(np.float64(342.1200085931923), np.float64(72.58734917956788))

In [9]:
results["eval()"] = benchmark_model(eval_mode, model, encodings, fn_setup=model.eval)
results["eval()"]

(np.float64(267.0232794259209), np.float64(67.45502704105296))

In [10]:
results["eval() + no_grad()"] = benchmark_model(
    eval_mode_and_no_grad, model, encodings, fn_setup=model.eval
)
results["eval() + no_grad()"]

(np.float64(258.1108105671592), np.float64(58.56660747233644))

In [11]:
results["eval() + inference_mode()"] = benchmark_model(
    eval_mode_and_inference_mode, model, encodings, fn_setup=model.eval
)
results["eval() + inference_mode()"]

(np.float64(250.13928243308328), np.float64(65.17022645532369))

In [12]:
print_benchmark_results(results, baseline="no optimization")

mode                        mean [ms]  std [ms]  speedup
no optimization                342.12     72.59    1.00x
eval()                         267.02     67.46    1.28x
eval() + no_grad()             258.11     58.57    1.33x
eval() + inference_mode()      250.14     65.17    1.37x


## 2. PyTorch model compilation

In [13]:
model.eval()
model_compiled = torch.compile(model)

In [14]:
start = time.perf_counter()
eval_mode_and_inference_mode(model_compiled, encodings)
compile_time_s = time.perf_counter() - start

print(f"Compilation + warm-up: {compile_time_s:.1f} s")

Compilation + warm-up: 11.9 s


In [15]:
results["compiled + eval() + inference_mode()"] = benchmark_model(
    eval_mode_and_inference_mode, model_compiled, encodings
)
results["compiled + eval() + inference_mode()"]

(np.float64(236.82732875458896), np.float64(62.128412791978114))

In [16]:
print_benchmark_results(results, baseline="no optimization")

mode                                   mean [ms]  std [ms]  speedup
no optimization                           342.12     72.59    1.00x
eval()                                    267.02     67.46    1.28x
eval() + no_grad()                        258.11     58.57    1.33x
eval() + inference_mode()                 250.14     65.17    1.37x
compiled + eval() + inference_mode()      236.83     62.13    1.44x


## 3. Quantization

In [17]:
# Dynamic quantization works on CPU - the model is already there (device = "cpu").
model = model.to("cpu")

model_quantized = torch.ao.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
print(model_quantized)

/tmp/ipykernel_32137/2811956075.py:4: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quantized = torch.ao.quantization.quantize_dynamic(


MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (k): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (v): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (o): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (dropout): Dropout(p=0.1, inplace=F

In [18]:
MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)


def save_and_get_size_mb(model, path):
    torch.save(model.state_dict(), path)
    return os.path.getsize(path) / (1024**2)

In [19]:
sizes = {}

sizes["Original model"] = save_and_get_size_mb(model, MODEL_DIR / "model_fp32.pt")
sizes["Compiled model"] = save_and_get_size_mb(
    model_compiled, MODEL_DIR / "model_compiled.pt"
)
sizes["Quantized model"] = save_and_get_size_mb(
    model_quantized, MODEL_DIR / "model_quantized.pt"
)

baseline_size = sizes["Original model"]

In [20]:
print(f"{'model':<38}{'size [MB]':>10}{'compression':>13}")
for name, size in sizes.items():
    print(f"{name:<38}{size:>10.2f}{baseline_size / size:>12.2f}x")

model                                  size [MB]  compression
Original model                            417.73        1.00x
Compiled model                            417.73        1.00x
Quantized model                           173.10        2.41x


In [21]:
results_cpu = {}

results_cpu["Original model"] = benchmark_model(
    eval_mode_and_inference_mode, model, encodings
)
results_cpu["Quantized model"] = benchmark_model(
    eval_mode_and_inference_mode, model_quantized, encodings
)

In [22]:
print_benchmark_results(results_cpu, baseline="Original model")

mode              mean [ms]  std [ms]  speedup
Original model       255.87     71.46    1.00x
Quantized model      164.01     48.47    1.56x


## 4. GPU optimization strategies

In [23]:
torch._dynamo.reset()

device = "cuda"
model = model.to(device).eval()
encodings = encodings.to(device)

torch.cuda.empty_cache()
print("Model on", next(model.parameters()).device)

Model on cuda:0


In [24]:
texts = {
    "short": sentence_a,
    "medium": " ".join([sentence_a] * 4),
    "long": " ".join([sentence_a] * 16),
}

inputs_by_size = {}
for size_name, text in texts.items():
    inputs_by_size[size_name] = tokenizer(
        [text, text], return_tensors="pt", padding=True, truncation=True
    ).to(device)

# Our data is already on GPU, so there is no need to use `pin_memory`.

In [25]:
compile_variants = {
    "eager (no compile)": None,
    "compile (default)": {},
    'compile "max-autotune"': {"mode": "max-autotune"},
    'compile "max-autotune-no-cudagraphs"': {"mode": "max-autotune-no-cudagraphs"},
}

results_gpu = {size_name: {} for size_name in inputs_by_size}

for variant, compile_kwargs in compile_variants.items():
    torch._dynamo.reset()
    model_variant = (
        model if compile_kwargs is None else torch.compile(model, **compile_kwargs)
    )

    for size_name, inputs in inputs_by_size.items():
        results_gpu[size_name][variant] = benchmark_model(
            eval_mode_and_inference_mode, model_variant, inputs
        )

W1122 13:07:51.073000 32137 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


In [26]:
for size_name, res in results_gpu.items():
    print(f"\n=== Input: {size_name} ===")
    print_benchmark_results(res, baseline="compile (default)")


=== Input: short ===
mode                                   mean [ms]  std [ms]  speedup
eager (no compile)                          6.71      0.58    0.48x
compile (default)                           3.23      0.07    1.00x
compile "max-autotune"                      1.51      0.03    2.13x
compile "max-autotune-no-cudagraphs"        3.22      0.08    1.00x

=== Input: medium ===
mode                                   mean [ms]  std [ms]  speedup
eager (no compile)                          6.61      0.52    0.52x
compile (default)                           3.47      0.36    1.00x
compile "max-autotune"                      1.90      0.02    1.82x
compile "max-autotune-no-cudagraphs"        3.22      0.06    1.08x

=== Input: long ===
mode                                   mean [ms]  std [ms]  speedup
eager (no compile)                          6.68      0.48    0.84x
compile (default)                           5.64      0.05    1.00x
compile "max-autotune"                      5.41  

`max-autotune` sped up only the short input (1.46x, and about 1.0x for medium and long), while `max-autotune-no-cudagraphs` was practically equal to default.

## 4. Changing numerical precision

In [27]:
import torch

capability = torch.cuda.get_device_capability()
print(f"CUDA device capability: {capability}")

if capability >= (7, 0):
    print("Tensor Cores available: fast float16 supported.")
else:
    print("Tensor Cores not available: float16 may be slow or unsupported.")

CUDA device capability: (12, 0)
Tensor Cores available: fast float16 supported.


In [28]:
# .half() casts the module IN PLACE and returns the same object - let's deepcopy it.
model_half = copy.deepcopy(model).half()
outputs = model_half(
    inputs_by_size["short"]["input_ids"], inputs_by_size["short"]["attention_mask"]
)

In [29]:
outputs["last_hidden_state"].shape

torch.Size([2, 16, 768])

Some tests from the lab instructions:

In [30]:
model_fp32 = torch.nn.Linear(10, 1)
data_fp32 = torch.randn(100, 10)
labels_fp32 = torch.randn(100, 1)

print(f"Data type of model_fp32 parameters: {model_fp32.weight.dtype}")
print(f"Data type of data_fp32: {data_fp32.dtype}")
print(f"Data type of labels_fp32: {labels_fp32.dtype}")

output_fp32 = model_fp32(data_fp32)
loss_fn = torch.nn.MSELoss()
loss_fp32 = loss_fn(output_fp32, labels_fp32)

print(f"Loss fp32: {loss_fp32.item()}")

Data type of model_fp32 parameters: torch.float32
Data type of data_fp32: torch.float32
Data type of labels_fp32: torch.float32
Loss fp32: 1.3356655836105347


In [31]:
model_fp16 = model_fp32.half()
data_fp16 = data_fp32.half()
labels_fp16 = labels_fp32.half()

print(f"Data type of model_fp16 parameters: {model_fp16.weight.dtype}")
print(f"Data type of data_fp16: {data_fp16.dtype}")
print(f"Data type of labels_fp16: {labels_fp16.dtype}")

output_fp16 = model_fp16(data_fp16)
loss_fp16 = loss_fn(output_fp16.float(), labels_fp16.float())

print(f"Loss fp16: {loss_fp16.item()}")

Data type of model_fp16 parameters: torch.float16
Data type of data_fp16: torch.float16
Data type of labels_fp16: torch.float16
Loss fp16: 1.3355460166931152


Model inference time of different models' precision:

In [32]:
assert next(model.parameters()).dtype == torch.float32, (
    "model was cast to half - reload it"
)
assert next(model_half.parameters()).dtype == torch.float16
assert not model.training and not model_half.training


def autocast_inference(model, inputs):
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        model(**inputs)

In [33]:
precision_variants = {
    "float32": (model, eval_mode_and_inference_mode),
    "float16 (manual)": (model_half, eval_mode_and_inference_mode),
    "autocast": (model, autocast_inference),
}

results_precision = {size_name: {} for size_name in inputs_by_size}

for size_name, inputs in inputs_by_size.items():
    for variant, (model_variant, fn) in precision_variants.items():
        results_precision[size_name][variant] = benchmark_model(
            fn, model_variant, inputs
        )

In [34]:
for size_name, res in results_precision.items():
    print(f"\n=== Input: {size_name} ===")
    print_benchmark_results(res, baseline="float32")


=== Input: short ===
mode               mean [ms]  std [ms]  speedup
float32                 6.72      3.23    1.00x
float16 (manual)        7.07      0.91    0.95x
autocast                9.33      0.89    0.72x

=== Input: medium ===
mode               mean [ms]  std [ms]  speedup
float32                 6.32      0.09    1.00x
float16 (manual)        6.61      0.12    0.96x
autocast                9.17      0.53    0.69x

=== Input: long ===
mode               mean [ms]  std [ms]  speedup
float32                 6.45      0.09    1.00x
float16 (manual)        6.66      0.30    0.97x
autocast                8.76      0.43    0.74x


My older GTX 1660 Ti card does not have Tensor Cores, so using FP32 for inference results in significantly slower performance. Let's leave this experiment as it is.

Autocast seems like a promising option, as it automatically identifies operations that require higher precision while performing less demanding operations in lower precision. I think I'll explore this approach further in the future.

In [35]:
torch.cuda.empty_cache()

## 5. ONNX

In [36]:
print("Model output keys: ", model(**encodings).keys())

Model output keys:  odict_keys(['last_hidden_state', 'pooler_output'])


Examples of using ONNX from lab instructions:

In [37]:
import torch.onnx

model_cpu = model.eval().cpu()

# Example input for tracking (for onnx export).
sample_input = tokenizer(
    "This is a sample input text for ONNX export.",
    padding=True,
    truncation=True,
    return_tensors="pt",
)

# Export to ONNX format.
export = torch.onnx.export(
    model_cpu,
    (sample_input["input_ids"], sample_input["attention_mask"]),
    "model.onnx",
    opset_version=17,
    input_names=["input_ids", "attention_mask"],
    output_names=["output"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size"},
    },
)

/tmp/ipykernel_32137/482482915.py:14: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  export = torch.onnx.export(
W1122 13:09:21.951000 32137 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MPNetModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MPNetModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/venv/main/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/workspace/mlops/.venv/lib/python3.12/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


In [38]:
import onnxruntime as ort

options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ort_session = ort.InferenceSession(
    "model.onnx", sess_options=options, providers=["CPUExecutionProvider"]
)

# Prepare input data.
sample_input = tokenizer(
    "This is a sample input text for ONNX inference.",
    padding=True,
    truncation=True,
    return_tensors="np",
)


# Create input dictionary, in same format as during export.
inputs_onnx = {
    "input_ids": sample_input["input_ids"],
    "attention_mask": sample_input["attention_mask"],
}

# Run inference.
outputs_onnx = ort_session.run(None, inputs_onnx)

----

Cold start and optimization modes benchmarks:

In [39]:
# Create optimized (offline-mode) model:
options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
options.optimized_model_filepath = "model_optimized.onnx"

ort.InferenceSession(
    "model.onnx", sess_options=options, providers=["CPUExecutionProvider"]
)

In [40]:
def create_session(path, level):
    options = ort.SessionOptions()
    options.graph_optimization_level = level
    return ort.InferenceSession(
        path, sess_options=options, providers=["CPUExecutionProvider"]
    )


def cold_start(path, level):
    create_session(path, level).run(None, inputs_onnx)


# Skip optimization and only load the model - pass ORT_DISABLE_ALL.
offline = ("model_optimized.onnx", ort.GraphOptimizationLevel.ORT_DISABLE_ALL)
online = ("model.onnx", ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED)

In [41]:
results_onnx_cold = {}

results_onnx_cold["online"] = benchmark(
    partial(cold_start, *online), device="cpu", n_runs=100, n_warmup=0
)
results_onnx_cold["offline"] = benchmark(
    partial(cold_start, *offline), device="cpu", n_runs=100, n_warmup=0
)

print_benchmark_results(results_onnx_cold, baseline="online")

mode      mean [ms]  std [ms]  speedup
online       882.81    126.57    1.00x
offline      876.59    133.15    1.01x


In [42]:
session_online = create_session(*online)
session_offline = create_session(*offline)

results_onnx = {}

results_onnx["online"] = benchmark(
    partial(session_online.run, None, inputs_onnx), device="cpu", n_runs=100
)
results_onnx["offline"] = benchmark(
    partial(session_offline.run, None, inputs_onnx), device="cpu"
)

print_benchmark_results(results_onnx, baseline="online")

mode      mean [ms]  std [ms]  speedup
online       155.13     45.85    1.00x
offline      180.01     46.76    0.86x


## 6. Docker images comparison - compiled PyTorch vs ONNX Runtime

In [ ]:
import json
import subprocess
import urllib.request

IMAGES = {"pytorch": "lab7-torch", "onnx": "lab7-onnx"}


def sh(*args):
    return subprocess.run(
        args, check=True, capture_output=True, text=True
    ).stdout.strip()


for name, tag in IMAGES.items():
    sh("docker", "build", "-f", f"docker/{name}.Dockerfile", "-t", tag, ".")

In [ ]:
sizes_docker = {}
for name, tag in IMAGES.items():
    size_bytes = int(sh("docker", "image", "inspect", tag, "--format", "{{.Size}}"))
    sizes_docker[name] = size_bytes / 1024**2

print(f"{'image':<10}{'size [MB]':>10}{'compression':>13}")
for name, size in sizes_docker.items():
    print(f"{name:<10}{size:>10.2f}{sizes_docker['pytorch'] / size:>12.2f}x")

In [ ]:
ports = {"pytorch": 8001, "onnx": 8002}


def post(port):
    request = urllib.request.Request(
        f"http://localhost:{port}/embed",
        data=json.dumps({"text": sentence_a}).encode(),
        method="POST",
    )
    urllib.request.urlopen(request).read()


def wait_until_ready(port, timeout_s=300):
    start = time.perf_counter()
    while True:
        try:
            post(port)
            return time.perf_counter() - start
        except OSError:
            if time.perf_counter() - start > timeout_s:
                raise
            time.sleep(1)


startup_docker = {}
results_docker = {}

for name, tag in IMAGES.items():
    container = sh("docker", "run", "-d", "--rm", "-p", f"{ports[name]}:8000", tag)
    try:
        startup_docker[name] = wait_until_ready(ports[name])
        results_docker[name] = benchmark(
            partial(post, ports[name]), device="cpu", n_runs=100
        )
    finally:
        sh("docker", "stop", container)

In [ ]:
print(f"Startup until first response [s]: {startup_docker}")

In [ ]:
print_benchmark_results(results_docker, baseline="pytorch")

Startup until first response [s]: {'pytorch': 44.501831305999076, 'onnx': 4.060570248999284}.

I'm currently unable to run the Docker-based implementation. I was able to do so previously, but now my old PC cannot handle it. I ran the GPU-related parts of this notebook on a remote machine equipped with an RTX 5060 Ti, but unfortunately, that environment does not allow Docker containers to be run.

As far as I can remember, inference using the ONNX Docker container was significantly faster, potentially up to 6 times faster than the alternative approach.